Import all necessary libraries, covering parsing logs,labeling, analysis, recons

In [2]:
import pandas as pd
from pathlib import Path

# -------- parsing ----------
from parsing.syslog import parse_syslog_csv
from parsing.auditd import parse_audit_log
from parsing.auth import parse_auth_log
from parsing.zeek import (
    parse_conn_log, parse_dns_log, parse_http_log, 
    parse_files_log, parse_ssl_log, parse_ssh_log
)
from parsing.suricata import parse_suricata_eve
from parsing.azure_wins import (
    parse_azure_conn, parse_azure_process, parse_azure_security,
    parse_azure_events, parse_azure_port, parse_azure_perf
)
from parsing.tracee import parse_tracee_ndjson

# ---------- labeling / analysis / recons ----------
from labeling.tagger import tag_steps
from analysis.coverage import step_coverage
from analysis.failure_taxonomy import failure_taxonomy 
from analysis.metrics import compute_metrics
from analysis.ambig import ambiguity

from recons.event_graph import build_event_graph
from recons.chain_builder import reconstruct_chain_detailed
# --- scenarios config ---
from scenarios import SCENARIOS

LOG_ROOT = Path("logs")


define the mapping from source -> parser

In [ ]:
from typing import Callable, List, Tuple, Dict

def _find_files(log_root: Path, patterns: List[str]) -> List[Path]:
    ''' recursively find files matching any of the given patterns in log_root

    '''
    hits = []
    for pat in patterns:
        hits.extend(log_root.rglob(pat) if not pat.startswith("**/") else log_root.rglob(pat[3:]))
    # remove duplicates while preserving order
    uniq = sorted(set([p for p in hits if p.is_file()]))
    return uniq


def _parse_many(files: List[Path], parser: Callable[[str], pd.DataFrame], *, source: str, source_kind: str) -> pd.DataFrame:
    ''' Multiple files are parsed and merged using the same parser, and source fields are added.
    
    '''
    frames = []
    for p in files:
        try:
            df = parser(str(p))
        except Exception:
            print(f"Warning: failed to parse {p} with {parser.__name__}")
            continue
        if df is None or len(df) == 0:
            continue
        df = df.copy()
        df["source"] = source
        df["source_kind"] = source_kind
        df["source_file"] = str(p)
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def _parse_prefixed_components(
    log_root: Path,
    *,
    prefix: str, # "zeek"/"azure"
    component_parsers: Dict[str, Callable[[str], pd.DataFrame]],
    source: str,
) -> pd.DataFrame:
    '''
    recursively find and parse log files for components with given prefix
    '''
    frames = []
    for comp, fn in component_parsers.items():
        patterns = [
            f"{prefix}_{comp}.csv",
            f"{prefix}_{comp}_*.csv",
            f"{prefix}_{comp}*.csv",
        ]
        files = _find_files(log_root, patterns)
        df = _parse_many(files, fn, source=source, source_kind=f"{source}_{comp}")
        if len(df):
            frames.append(df)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def parse_source(source_key: str, log_root: Path) -> pd.DataFrame:
    '''
    return unified source dataframe for a given source_key
    args:
        log_root: Path to scenario log root
    '''
    # --- syslog ---
    if source_key == "syslog":
        files = _find_files(log_root, ["syslog.csv", "syslog*.csv"])
        return _parse_many(files, parse_syslog_csv, source="syslog", source_kind="syslog")
    
    # --- auditd ---
    if source_key == "auditd":
        files = _find_files(log_root, ["audit.log", "auditd*.log"])
        return _parse_many(files, parse_audit_log, source="auditd", source_kind="auditd")

    # --- auth ---
    if source_key == "auth":
        files = _find_files(log_root, ["auth.log", "auth*.log"])
        return _parse_many(files, parse_auth_log, source="auth", source_kind="auth")

    # ----- suricate -----
    if source_key == "suricata":
        files = _find_files(log_root, ["eve.json", "eve*.json"])
        return _parse_many(files, parse_suricata_eve, source="suricata", source_kind="suricata")

    # ----- tracee -------
    if source_key == "tracee":
        files = _find_files(log_root, ["tracee-events.json"])
        return _parse_many(files, parse_suricata_eve, source="tracee", source_kind="tracee")


    # --- zeek (multi-log) ---
    if source_key == "zeek":
        zeek_parsers: List[Tuple[str, Callable[[str], pd.DataFrame]]] = [
            ("conn",  parse_conn_log),
            ("dns",   parse_dns_log),
            ("http",  parse_http_log),
            ("files", parse_files_log),
            ("ssh",   parse_ssh_log),
            ("ssl",   parse_ssl_log),
        ]
        return _parse_prefixed_components(
            log_root,
            prefix="zeek",
            component_parsers=zeek_parsers,
            source="zeek",
        )

    # --- azure windows ---
    if source_key == "azure_wins":
        azure_parsers = {
            "conn":     parse_azure_conn,
            "process":  parse_azure_process,
            "security": parse_azure_security,
            "events":   parse_azure_events,
            "syslog":  parse_syslog_csv,
        }

        return _parse_prefixed_components(
        log_root,
        prefix="azure",
        component_parsers=azure_parsers,
        source="azure_wins",
        )

run pipeline (tag + metrics + chain) with given sources

In [ ]:
def run_pipeline_for_sources(enabled_sources, log_root: Path):
    # 1) parsing
    frames = []

    for s in enabled_sources:
        df = parse_source(s, log_root)
        if df is not None and len(df) > 0:
            df['source'] = s
            frames.append(df)
    
    events = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    # 2) tagging
    tagged = tag_steps(events) if len(events) else events

    # 3) analytics
    cov = step_coverage(tagged) if len(tagged) else {}
    fails = failure_taxonomy(tagged) if len(tagged) else {}
    amb = ambiguity(tagged) if len(tagged) else {}
    d = reconstruct_chain_detailed(g, tagged.dropna(subset=["step"]))
    chain_steps = d["steps"]
    metrics = compute_metrics(tagged, chain=chain_steps, max_gap_seconds=600)    

    # 4) reconstruction
    chain = []
    if len(tagged) and "step" in tagged.columns:
        g = build_event_graph(tagged)
        chain = reconstruct_chain_detailed(g, tagged.dropna(subset=["step"]))['steps']

    # 5) reconstruction summary
    observed_steps = []
    if len(tagged) and "step" in tagged.columns:
        observed_steps = sorted([x for x in tagged["step"].dropna().unique().tolist()])
    
    chain_steps = []
    if chain:
        if isinstance(chain[0], str):
            chain_steps = chain
        elif isinstance(chain[0], dict) and "step" in chain[0]:
            chain_steps = [x["step"] for x in chain]
        else:
            chain_steps = chain

    recon = {
        "n_events": int(len(events)) if len(events) else 0,
        "n_tagged_steps": int(len(observed_steps)),
        "n_chain_steps": int(len(chain_steps)),
        "reconstructability": (len(chain_steps) / len(observed_steps)) if observed_steps else 0.0,
        "observed_steps": observed_steps,
        "chain_steps": chain_steps,
    }

    return {
        "events": events,
        "tagged": tagged,
        "coverage": cov,
        "failures": fails,
        "ambiguity": amb,
        "metrics": metrics,
        "chain": chain,
        "recon": recon,
    }
    

### comparison between single source vs multiple sources

- multiple sources: all sources from one scenario
- single source: every source in one scenarios separately runs once, find failure mode and missing part
- output one summary

In [ ]:
def get_expected_sources_for_scenario(sc) -> list:
    # sc.expected_sources is dict[str,bool]
    return [k for k, v in sc.expected_sources.items() if v]

def analyze_scenario(scenario_key: str, log_root: Path):
    # extract scenario config
    sc = SCENARIOS[scenario_key]
    expected_sources = get_expected_sources_for_scenario(sc)

    # ---- multi-source run -----
    multi = run_pipeline_for_sources(expected_sources, log_root)

    # --- single-source runs ---
    single_rows = []
    single_outputs = {}
    for s in expected_sources:
        out = run_pipeline_for_sources([s], log_root)
        single_outputs[s] = out
        single_rows.append({
            "scenario": sc.id,
            "name": sc.name,
            "mode": "single",
            "source_set": s,
            **out["recon"],
            "failure_keys": sorted(list(out["failures"].keys())) if isinstance(out["failures"], dict) else [],
        })
    
    # best single-source by recon
    single_df = pd.DataFrame(single_rows)
    if len(single_df):
        best_single = single_df.sort_values(["reconstructability", "n_chain_steps", "n_tagged_steps"], ascending=False).head(1)
        best_single_row = best_single.iloc[0].to_dict()
    else:
        best_single_row = {}

    # multi row
    multi_row = {
        "scenario": sc.id,
        "name": sc.name,
        "mode": "multi",
        "source_set": "+".join(expected_sources),
        **multi["recon"],
        "failure_keys": sorted(list(multi["failures"].keys())) if isinstance(multi["failures"], dict) else [],
    }

    # --- Gap explanation helpers (for Q2/Q3) ---
    # Which steps appear in multi-source scenarios but are generally missing in single-source scenarios? (This demonstrates the complementarity of multi-source scenarios)
    multi_steps = set(multi["recon"]["observed_steps"])
    step_presence = {}
    for s, out in single_outputs.items():
        step_presence[s] = set(out["recon"]["observed_steps"])

    step_missing_in_single = {}
    for st in sorted(multi_steps):
        missing_sources = [s for s in expected_sources if st not in step_presence.get(s, set())]
        if missing_sources:
            step_missing_in_single[st] = missing_sources

    # Which failures are most frequent in single-source scenarios but mitigated by multi-source scenarios? (Reflecting mitigation)

    # Here, we simply use the difference of the key in failure_taxonomy; you can further expand the counts.
    def failure_keys(x): 
        return set(x.keys()) if isinstance(x, dict) else set()

    multi_fail = failure_keys(multi["failures"])
    single_fail_union = set()
    for s in expected_sources:
        single_fail_union |= failure_keys(single_outputs[s]["failures"])

    mitigated_failures = sorted(list(single_fail_union - multi_fail))

    summary = {
        "scenario": sc.id,
        "name": sc.name,
        "expected_sources": expected_sources,
        "multi_reconstructability": multi["recon"]["reconstructability"],
        "best_single_source": best_single_row.get("source_set"),
        "best_single_reconstructability": best_single_row.get("reconstructability", 0.0),
        "multi_minus_best_single": multi["recon"]["reconstructability"] - best_single_row.get("reconstructability", 0.0),
        "steps_missing_in_single": step_missing_in_single,   
        "failures_mitigated_by_multi": mitigated_failures, 

        # ---- CSA-2 placeholders (leave for later) ----
        "expected_payload_ttps": None,
        "expected_c2_ttps": None,
    }

    return {
        "multi": multi,
        "single": single_outputs,
        "single_df": single_df,
        "multi_row": multi_row,
        "summary": summary,
    }



In [ ]:
def pretty_three_questions(summary: dict) -> str:
    sc = summary["scenario"]
    name = summary["name"]
    multi_r = summary["multi_reconstructability"]
    best_s = summary["best_single_source"]
    best_r = summary["best_single_reconstructability"]
    delta = summary["multi_minus_best_single"]

    # Q1: end-to-end reconstructability
    q1 = (
        f"[Q1] {sc} ({name}) — End-to-end reconstructability under multi-source telemetry: "
        f"reconstructability={multi_r:.2f}. "
        f"Best single-source baseline ({best_s})={best_r:.2f} "
        f"(multi − best_single = {delta:+.2f})."
    )

    # Q2: where/why single-source fails (proxy via steps missing in single-source runs)
    missing = summary["steps_missing_in_single"]
    top_missing = list(missing.items())[:8]  # keep it readable

    if top_missing:
        q2_lines = [
            f"  - step={st} is absent in single-source runs for: {srcs}"
            for st, srcs in top_missing
        ]
        q2 = (
            "[Q2] Single-source failure points — examples of steps observable in the multi-source run "
            "but missing under individual sources:\n"
            + "\n".join(q2_lines)
        )
    else:
        q2 = (
            "[Q2] Single-source failure points — no clear step-level omissions detected "
            "(this can happen when the chain is short, when sources have overlapping visibility, "
            "or when labeling evidence is sparse)."
        )

    # Q3: how multi-source mitigates failures (proxy via failure categories that disappear in multi-source)
    mitigated = summary["failures_mitigated_by_multi"]

    if mitigated:
        q3 = (
            "[Q3] Multi-source mitigation — failure categories observed in at least one single-source run "
            "but not present in the multi-source run:\n"
            + "\n".join([f"  - {x}" for x in mitigated[:10]])
        )
    else:
        q3 = (
            "[Q3] Multi-source mitigation — no clear difference in failure keys "
            "(consider refining this with step-level failure counts and/or explicit observability-gap signals)."
        )

    return q1 + "\n" + q2 + "\n" + q3


# ---- run ----
scenario_keys = list(SCENARIOS.keys())  # or specify manually
all_rows = []
all_summaries = {}

for k in scenario_keys:
    res = analyze_scenario(k, LOG_ROOT)
    all_rows.append(res["multi_row"])
    # also include single-source rows for comparison
    if len(res["single_df"]):
        all_rows.extend(res["single_df"].to_dict("records"))
    all_summaries[k] = res["summary"]

result_df = pd.DataFrame(all_rows)
result_df.sort_values(["scenario", "mode", "reconstructability"], ascending=[True, True, False], inplace=True)
result_df